# California DOI PDF scraper

Crawls [California Department of Insurance](https://www.insurance.ca.gov/) (same host only), follows `robots.txt` rules for `User-agent: *`, and downloads PDFs into `data/california/`.

**Note:** `robots.txt` blocks the default `Python-urllib` client entirely; this notebook uses `httpx` with a custom `User-Agent`. With `USE_SITEMAP = True`, PDFs are taken from the site’s [`sitemap.xml`](https://www.insurance.ca.gov/sitemap.xml) (~2,800+ direct PDF links) **plus** any extra PDFs found while crawling HTML. `MAX_PAGES` limits **HTML** fetches only; the queue must empty (set `MAX_PDF_DOWNLOADS` to cap PDFs). A full sitemap run can take ~20–40 minutes depending on `REQUEST_DELAY_SEC`.

In [ ]:
from __future__ import annotations

import hashlib
import re
import time
import xml.etree.ElementTree as ET
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse, urlunparse

import httpx
from bs4 import BeautifulSoup

BASE = "https://www.insurance.ca.gov"
STATE = "california"
OUT_DIR = Path("data") / STATE
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-CaliforniaDOI-PDFScraper/1.0 (educational research; respects robots.txt)"
HEADERS = {"User-Agent": USER_AGENT, "Accept": "text/html,application/xhtml+xml,application/pdf;q=0.9,*/*;q=0.8"}

# HTML crawl budget (PDFs from sitemap + links still run after this)
MAX_PAGES = 2500
MAX_DEPTH = 6
REQUEST_DELAY_SEC = 0.35
TIMEOUT = 90.0
MAX_PDF_DOWNLOADS: int | None = None

# ~2,800+ PDFs are listed in the official sitemap (far more than link-only crawl finds)
USE_SITEMAP = True
SITEMAP_URL = urljoin(BASE, "/sitemap.xml")

INDEX_PATH = OUT_DIR / "_download_index.tsv"
VISITED_PATH = OUT_DIR / "_visited_urls.txt"

In [ ]:
def parse_robots_star_disallow(robots_txt: str) -> list[str]:
    disallows: list[str] = []
    in_star = False
    for line in robots_txt.splitlines():
        raw = line.split("#", 1)[0].strip()
        if not raw:
            continue
        low = raw.lower()
        if low.startswith("user-agent:"):
            ua = raw.split(":", 1)[1].strip()
            in_star = ua == "*"
        elif in_star and low.startswith("disallow:"):
            path = raw.split(":", 1)[1].strip()
            if path:
                disallows.append(path)
    return disallows


def path_robots_allowed(url: str, disallows: list[str]) -> bool:
    p = urlparse(url)
    path = p.path or "/"
    return not any(path.startswith(d) for d in disallows)


def canonicalize_http_url(url: str, base: str) -> str | None:
    joined = urljoin(base, url)
    parts = urlparse(joined)
    if parts.scheme not in ("http", "https"):
        return None
    host = (parts.hostname or "").lower()
    if host not in ("www.insurance.ca.gov", "insurance.ca.gov"):
        return None
    path = parts.path or "/"
    if path.endswith("/index.htm") or path.endswith("/index.html"):
        path = path.rsplit("/", 1)[0] + "/"
    norm = urlunparse(("https", "www.insurance.ca.gov", path, "", parts.query, ""))
    return norm


_HTML_EXT = (".cfm", ".htm", ".html", ".shtml", ".asp", ".aspx")


def looks_like_html_url(url: str) -> bool:
    path = urlparse(url).path.lower()
    if path.endswith(".pdf") or ".pdf?" in path:
        return False
    if path.endswith(_HTML_EXT) or path.endswith("/"):
        return True
    base = Path(path).name
    return "." not in base


def sitemap_pdf_urls(client: httpx.Client, sitemap_url: str) -> list[str]:
    r = client.get(sitemap_url)
    r.raise_for_status()
    root = ET.fromstring(r.text)
    out: list[str] = []
    for elem in root.iter():
        if not elem.tag.endswith("loc") or not elem.text:
            continue
        u = elem.text.strip()
        if u.lower().split("?", 1)[0].endswith(".pdf"):
            out.append(u)
    return out


def safe_pdf_filename(url: str) -> str:
    name = Path(urlparse(url).path).name
    if not name or not name.lower().endswith(".pdf"):
        h = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]
        name = f"doc_{h}.pdf"
    name = re.sub(r"[^\w.\-]", "_", name)
    return name[:180]


def extract_links(html: str, page_url: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")
    out: list[str] = []
    for tag in soup.find_all("a", href=True):
        href = tag["href"].strip()
        if href.startswith(("javascript:", "mailto:", "#")):
            continue
        out.append(urljoin(page_url, href))
    for tag in soup.find_all(["iframe", "embed"], src=True):
        src = tag["src"].strip()
        if src:
            out.append(urljoin(page_url, src))
    return out

In [ ]:
def run_crawl() -> None:
    with httpx.Client(headers=HEADERS, timeout=TIMEOUT, follow_redirects=True) as client:
        r = client.get(urljoin(BASE, "/robots.txt"))
        r.raise_for_status()
        disallows = parse_robots_star_disallow(r.text)

        sm_pdfs: list[str] = []
        if USE_SITEMAP:
            sm_pdfs = sitemap_pdf_urls(client, SITEMAP_URL)

    print(f"Loaded {len(disallows)} Disallow rules for User-agent: *")
    if USE_SITEMAP:
        print(f"Sitemap: {len(sm_pdfs)} PDF URLs")

    visited: set[str] = set()
    downloaded: set[str] = set()
    priority = [
        BASE.rstrip("/") + "/",
        urljoin(BASE, "/0250-insurers/0300-insurers/0200-bulletins/index.cfm"),
        urljoin(BASE, "/0400-news/0200-studies-reports/index.cfm"),
        urljoin(BASE, "/0400-news/0100-press-releases/index.cfm"),
        urljoin(BASE, "/0200-industry/0000-industry-overview/index.cfm"),
        urljoin(BASE, "/0300-fraud/0100-fraud-division-overview/index.cfm"),
    ]
    queue: deque[tuple[str, int]] = deque()
    for u in priority:
        cu0 = canonicalize_http_url(u, BASE)
        if cu0 and path_robots_allowed(cu0, disallows):
            queue.append((cu0, 0))
    for u in reversed(sm_pdfs):
        cu0 = canonicalize_http_url(u, BASE)
        if cu0 and path_robots_allowed(cu0, disallows):
            queue.appendleft((cu0, 0))
    pages_fetched = 0

    with httpx.Client(headers=HEADERS, timeout=TIMEOUT, follow_redirects=True) as client:
        while queue:
            url, depth = queue.popleft()
            cu = canonicalize_http_url(url, BASE)
            if not cu or cu in visited:
                continue
            if not path_robots_allowed(cu, disallows):
                visited.add(cu)
                continue

            if urlparse(cu).path.lower().endswith(".pdf"):
                if MAX_PDF_DOWNLOADS is not None and len(downloaded) >= MAX_PDF_DOWNLOADS:
                    visited.add(cu)
                    continue
                visited.add(cu)
                time.sleep(REQUEST_DELAY_SEC)
                if cu in downloaded:
                    continue
                try:
                    pr = client.get(cu)
                    pr.raise_for_status()
                    if "pdf" not in (pr.headers.get("content-type") or "").lower() and pr.content[:4] != b"%PDF":
                        continue
                    fn = safe_pdf_filename(str(pr.url))
                    dest = OUT_DIR / fn
                    if dest.exists():
                        stem = dest.stem
                        dest = OUT_DIR / f"{stem}_{hashlib.sha256(str(pr.url).encode()).hexdigest()[:8]}.pdf"
                    dest.write_bytes(pr.content)
                    downloaded.add(cu)
                    with INDEX_PATH.open("a", encoding="utf-8") as f:
                        f.write(f"{pr.url}\t{dest.name}\n")
                    print(f"PDF: {dest.name}")
                except httpx.HTTPError as e:
                    print(f"Skip PDF {cu}: {e}")
                continue

            if pages_fetched >= MAX_PAGES:
                visited.add(cu)
                continue

            if depth > MAX_DEPTH:
                visited.add(cu)
                continue

            if not looks_like_html_url(cu):
                visited.add(cu)
                continue

            visited.add(cu)
            time.sleep(REQUEST_DELAY_SEC)

            try:
                resp = client.get(cu)
                resp.raise_for_status()
            except httpx.HTTPError as e:
                print(f"Skip page {cu}: {e}")
                continue

            ctype = (resp.headers.get("content-type") or "").lower()
            if "pdf" in ctype:
                final = str(resp.url)
                if final not in downloaded:
                    fn = safe_pdf_filename(final)
                    dest = OUT_DIR / fn
                    dest.write_bytes(resp.content)
                    downloaded.add(final)
                    with INDEX_PATH.open("a", encoding="utf-8") as f:
                        f.write(f"{final}\t{dest.name}\n")
                    print(f"PDF (content-type): {dest.name}")
                continue

            if "html" not in ctype and "cfm" not in ctype and "xhtml" not in ctype:
                continue

            pages_fetched += 1
            if pages_fetched % 25 == 0:
                print(f"… pages fetched: {pages_fetched}, queue: {len(queue)}")

            for link in extract_links(resp.text, str(resp.url)):
                nu = canonicalize_http_url(link, BASE)
                if not nu or not path_robots_allowed(nu, disallows):
                    continue
                if nu in visited:
                    continue
                if urlparse(nu).path.lower().endswith(".pdf"):
                    queue.appendleft((nu, depth))
                elif depth < MAX_DEPTH and looks_like_html_url(nu) and pages_fetched < MAX_PAGES:
                    queue.append((nu, depth + 1))

    VISITED_PATH.write_text("\n".join(sorted(visited)), encoding="utf-8")
    print(f"Done. Pages fetched: {pages_fetched}, PDFs saved: {len(downloaded)}, visited URLs: {len(visited)}")


if INDEX_PATH.exists():
    INDEX_PATH.unlink()
run_crawl()